# Example: comparing partitioners

Which partitioner should be the default for a new problem? This notebook runs the five partitioners with twenty
seeds on three samples of normal mixtures and compares the log-likelihood after a short EM and the time of a start.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

import time

from sklearn.cluster import AgglomerativeClustering, KMeans

from pysatl_mpest.initializers import (
    ClusterInitializer,
    InitializationError,
    KMeansPartitioner,
    QuantilePartitioner,
    RandomCentersPartitioner,
    SklearnPartitioner,
)

## Samples

- **separated** — three distant modes of equal size;
- **overlapping** — three modes closer than two standard deviations;
- **unequal** — one wide mode and two narrow ones close to each other.

In [ ]:
samples = {
    "separated": MixtureModel([normal(mu=m, sigma=1) for m in (-10, 0, 10)]),
    "overlapping": MixtureModel([normal(mu=m, sigma=1.5) for m in (-3, 0, 3)]),
    "unequal": MixtureModel(
        [normal(mu=0, sigma=3), normal(mu=15, sigma=0.3), normal(mu=17, sigma=0.3)], weights=[0.7, 0.15, 0.15]
    ),
}
data = {name: truth.generate(size=600, random_state=1) for name, truth in samples.items()}

fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))
for ax, (name, truth) in zip(axes, samples.items()):
    plot_mixtures(ax, data[name], {name: truth})
plt.show()

## Benchmark

In [ ]:
partitioners = {
    "k-means": KMeansPartitioner,
    "quantile": QuantilePartitioner,
    "random centers": RandomCentersPartitioner,
    "sklearn KMeans": lambda: SklearnPartitioner(KMeans(n_init=1)),
    "agglomerative": lambda: SklearnPartitioner(AgglomerativeClustering()),
}
scores = {sample: {} for sample in samples}
times = {name: [] for name in partitioners}
for sample, X in data.items():
    for name, make in partitioners.items():
        values = []
        for seed in range(20):
            began = time.perf_counter()
            try:
                start = ClusterInitializer(make()).initialize(X, [normal] * 3, random_state=seed)
            except InitializationError:
                continue
            times[name].append(time.perf_counter() - began)
            values.append(em(10).fit(X, start).loglikelihood(X))
        scores[sample][name] = values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (sample, by_partitioner) in zip(axes, scores.items()):
    ax.boxplot(list(by_partitioner.values()), tick_labels=list(by_partitioner))
    ax.axhline(samples[sample].loglikelihood(data[sample]), color="black", linestyle="--", label="true mixture")
    ax.set_title(sample)
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
plt.show()

for name, values in times.items():
    print(f"{name:15} mean time of a start {1000 * np.mean(values):.1f} ms")

## What the numbers say

- On **separated** modes every deterministic partitioner is right every time; random centers sometimes put two
  centers into one mode, and the short EM does not recover.
- On **overlapping** modes all of them end close to each other: the start matters less than the estimator.
- On **unequal** modes the quantile split, scikit-learn k-means and agglomerative clustering always cut the wide mode
  in two and merge the narrow ones; k-means does the same on almost every seed. Only a few draws — mostly of random
  centers — find the right structure.

No partitioner wins everywhere. Deterministic ones are good single starts; random centers are a cheap source of
different starts for a multistart.

In [ ]:
unequal = scores["unequal"]
assert max(unequal["random centers"]) > max(unequal["quantile"]) + 50
assert np.ptp(scores["separated"]["quantile"]) == 0